In [2]:
!pip install pandas openpyxl xlrd

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
'''SHEFI NEW PO - ALL COLUMNS CORRECTLY EXTRACTED AND SAVED TO EXCEL'''

import pandas as pd
import re
from datetime import datetime
import pdfplumber

def format_date(date_str):
    """Convert M/D/YYYY to DD-Mon-YY (e.g. 3/27/2026 -> 27-Mar-26)"""
    try:
        return datetime.strptime(date_str.strip(), '%m/%d/%Y').strftime('%d-%b-%y')
    except Exception:
        return date_str.strip()

def extract_shefi_po(pdf_path):
    """
    Extract all PO data page-by-page so each item carries its own correct page number.
    Correctly maps all 17 columns per the specification.
    """
    all_rows = []
    item_counter = 0   # continuous across pages

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text() or ''
            lines = [ln.strip() for ln in page_text.split('\n')]

            # ── Header extraction (per page) ────────────────────────────────
            hdr = {
                'Customer': 'SHEFI DIAMONDS, INC',
                'Order#': '', 'Page#': '', 'PO#': '',
                'Date': '', 'Due Date': '', 'Cancel Date': '',
                'Ref': '', 'Vendor#': '', 'Ship Via': ''
            }

            for line in lines:
                m = re.search(r'Order #:\s*(\d+)', line)
                if m:
                    hdr['Order#'] = m.group(1)

                m = re.search(r'Page #:\s*(\d+\s+of\s+\d+)', line)
                if m:
                    hdr['Page#'] = m.group(1)

                m = re.search(r'P\.O\. #:\s*(\S+)', line)
                if m:
                    hdr['PO#'] = m.group(1)

                # Combined date extraction to avoid mismatching Due/Cancel Date
                m = re.search(
                    r'Date:\s*(\d+/\d+/\d+)\s+Due Date:\s*(\d+/\d+/\d+)\s+Cancel Date:\s*(\d+/\d+/\d+)',
                    line
                )
                if m:
                    hdr['Date']        = format_date(m.group(1))
                    hdr['Due Date']    = format_date(m.group(2))
                    hdr['Cancel Date'] = format_date(m.group(3))

                m = re.search(r'Reference:\s*(\S+)', line)
                if m:
                    hdr['Ref'] = m.group(1)

                m = re.search(r'Vendor #:(\S+)', line)
                if m:
                    hdr['Vendor#'] = m.group(1)

                m = re.search(r'Ship Via:\s*(\S+)', line)
                if m:
                    hdr['Ship Via'] = m.group(1)

            # ── Item extraction (per page) ───────────────────────────────────
            current_item   = None
            current_cat    = None
            expect_item    = False

            # Lines that qualify as category headers (e.g. "LGD Anniversary:")
            # They start with a letter, contain text, and end with ":"
            CAT_PATTERN = re.compile(r'^[A-Za-z][A-Za-z0-9 ]+:$')
            # Keywords that signal a description continuation line
            DESC_KEYWORDS = ('Rd.', 'Ctw', 'Lab', 'Grown', 'Diamond')

            for line in lines:
                # ── Category line ────────────────────────────────────────────
                if CAT_PATTERN.match(line) and not any(
                    kw in line for kw in ('Order', 'Page', 'P.O.', 'Ref', 'Ship', 'Grand', 'Right')
                ):
                    if current_item:
                        # Finalise description and save previous item
                        extras = current_item.pop('_extra', [])
                        current_item['Description'] += (' ' + ' '.join(extras)) if extras else ''
                        all_rows.append(current_item)
                        current_item = None
                    current_cat = line.rstrip(':').strip()
                    expect_item = True
                    continue

                # ── Item data line ────────────────────────────────────────────
                if expect_item:
                    m = re.match(r'^(\d+)\s+([A-Z][A-Z0-9]+)\s+(.+)', line)
                    if m:
                        item_counter += 1
                        item_no = m.group(2)
                        rest    = m.group(3)
                        words   = rest.split()

                        # --- Vendor Item # -----------------------------------
                        # Present when the first word of `rest` is an alphanumeric
                        # product code (not a metal type like 14KW/14KY).
                        # Pattern: starts with 2 uppercase letters, >= 8 chars total.
                        vendor_item = ''
                        w_off = 0
                        if (words
                                and not re.match(r'^14K', words[0])
                                and re.match(r'^[A-Z]{2}', words[0])
                                and len(words[0]) >= 8):
                            vendor_item = words[0]
                            w_off = 1   # metal description starts one word later

                        # --- Metal description (e.g. "14KW Shared Prong") ----
                        metal_desc = ' '.join(words[w_off: w_off + 3]) if len(words) >= w_off + 3 else ''

                        # --- Stone count on same line (e.g. "11" before size) -
                        # Appears as an integer BETWEEN "Prong" and the decimal size.
                        stone_on_line = ''
                        idx_after_metal = w_off + 3
                        if len(words) > idx_after_metal:
                            cand = words[idx_after_metal]
                            if re.match(r'^\d+$', cand):
                                # Verify next word is the decimal size (e.g. 6.5)
                                if (len(words) > idx_after_metal + 1
                                        and re.match(r'^\d+\.\d+$', words[idx_after_metal + 1])):
                                    stone_on_line = cand

                        # --- Size and Quantity --------------------------------
                        # Always: <decimal_size> <qty> 0.0000  (e.g. "6.5 1 0.0000")
                        sq = re.search(r'(\d+\.\d+)\s+(\d+)\s+0\.0000', rest)
                        size_val = sq.group(1) if sq else ''
                        qty_val  = sq.group(2) if sq else ''

                        # --- Build initial description -----------------------
                        desc_parts = [current_cat or '', metal_desc]
                        if stone_on_line:
                            desc_parts.append(stone_on_line)
                        desc = ' '.join(p for p in desc_parts if p)

                        current_item = dict(hdr)
                        current_item.update({
                            '#':             item_counter,
                            'Memo #':        '',
                            'Item #':        item_no,
                            'Vendor Item #': vendor_item,
                            'Description':   desc,
                            'Size':          size_val,
                            'Quantity':      qty_val,
                            '_extra':        []          # temp; removed before saving
                        })
                        expect_item = False
                        continue

                # ── Continuation lines (extra description text) ───────────────
                if current_item and line:
                    if any(kw in line for kw in DESC_KEYWORDS):
                        current_item['_extra'].append(line)

            # Save the last item on this page
            if current_item:
                extras = current_item.pop('_extra', [])
                current_item['Description'] += (' ' + ' '.join(extras)) if extras else ''
                all_rows.append(current_item)

    return all_rows


# ── Run extraction ────────────────────────────────────────────────────────────
pdf_file_path = r'C:\Users\Admin\Desktop\SHEFI_NEW_PO\PDPO#29626.pdf'

rows = extract_shefi_po(pdf_file_path)

COLUMNS = [
    'Customer', 'Order#', 'Page#', 'PO#', 'Date', 'Due Date', 'Cancel Date',
    'Ref', 'Vendor#', 'Ship Via', '#', 'Memo #', 'Item #', 'Vendor Item #',
    'Description', 'Size', 'Quantity'
]

df_shefi = pd.DataFrame(rows)
for col in COLUMNS:
    if col not in df_shefi.columns:
        df_shefi[col] = ''
df_shefi = df_shefi[COLUMNS]

# ── Save to Excel ─────────────────────────────────────────────────────────────
output_path = r'C:\Users\Admin\Desktop\SHEFI_NEW_PO\SHEFI_PO_29626_FINAL_hello.xlsx'
df_shefi.to_excel(output_path, index=False)

# ── Display results ───────────────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 70)

print(f"Total items extracted : {len(df_shefi)}")
print(f"Output file           : {output_path}")
print("=" * 90)
print(df_shefi.to_string(index=False))

# Verification summary
print("\n" + "=" * 90)
print("VERIFICATION")
print("=" * 90)
for _, row in df_shefi.iterrows():
    print(f"  #{row['#']:>2}  Item#={row['Item #']:<12}  "
          f"VendorItem#={str(row['Vendor Item #']):<12}  "
          f"Page#={row['Page#']:<7}  "
          f"Size={row['Size']:<5}  Qty={row['Quantity']}")
    print(f"       Desc: {row['Description']}")

df_shefi


CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


Total items extracted : 12
Output file           : C:\Users\Admin\Desktop\SHEFI_NEW_PO\SHEFI_PO_29626_FINAL_hello.xlsx
           Customer Order#  Page#   PO#      Date  Due Date Cancel Date   Ref Vendor# Ship Via  # Memo #     Item # Vendor Item #                                                          Description Size Quantity
SHEFI DIAMONDS, INC  29626 1 of 2 STOCK 27-Mar-26 27-Mar-26   27-Mar-26 STOCK    P533   BRINKS  1        LGD244910E               LGD Anniversary 14KW Shared Prong 11 Rd. 1/10 Ctw Lab Grown Diamonds  6.5        1
SHEFI DIAMONDS, INC  29626 1 of 2 STOCK 27-Mar-26 27-Mar-26   27-Mar-26 STOCK    P533   BRINKS  2        LGD244910D               LGD Anniversary 14KY Shared Prong 11 Rd. 1/10 Ctw Lab Grown Diamonds  6.5        1
SHEFI DIAMONDS, INC  29626 1 of 2 STOCK 27-Mar-26 27-Mar-26   27-Mar-26 STOCK    P533   BRINKS  3        LGD244925E                LGD Anniversary 14KW Shared Prong 11 Rd. 1/4 Ctw Lab Grown Diamonds  6.5        1
SHEFI DIAMONDS, INC  29626 1 

,Customer,Order#,Page#,PO#,Date,Due Date,Cancel Date,Ref,Vendor#,Ship Via,#,Memo #,Item #,Vendor Item #,Description,Size,Quantity
0,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,1,,LGD244910E,,LGD Anniversary 14KW Shared Prong 11 Rd. 1/10 Ctw Lab Grown Diamonds,6.5,1
1,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,2,,LGD244910D,,LGD Anniversary 14KY Shared Prong 11 Rd. 1/10 Ctw Lab Grown Diamonds,6.5,1
2,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,3,,LGD244925E,,LGD Anniversary 14KW Shared Prong 11 Rd. 1/4 Ctw Lab Grown Diamonds,6.5,1
3,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,4,,LGD244925D,,LGD Anniversary 14KY Shared Prong 11 Rd. 1/4 Ctw Lab Grown Diamonds,6.5,1
4,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,5,,LGD244933E,RG0004161E,LGD Anniversary 14KW Shared Prong 11 Rd. 1/3 Ctw Lab Grown Diamonds,6.5,1
5,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,6,,LGD244933D,RG0004161E,LGD Anniversary 14KY Shared Prong 11 Rd. 1/3 Ctw Lab Grown Diamonds,6.5,1
6,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,7,,LGD244950D,RG0004161F,LGD Anniversary 14KY Shared Prong 11 Rd. 1/2 Ctw Lab Grown Diamonds,6.5,1
7,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,8,,LGD244950E,RG0004161F,LGD Anniversary 14KW Shared Prong 11 Rd. 1/2 Ctw Lab Grown Diamonds,6.5,1
8,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,9,,LGD244975E,RG0004161G,LGD Anniversary 14KW Shared Prong 11 Rd. 3/4 Ctw Lab Grown Diamonds,6.5,1
9,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,10,,LGD244975D,RG0004161G,LGD Anniversary 14KY Shared Prong 11 Rd. 3/4 Ctw Lab Grown Diamonds,6.5,1
